# YOLO Candidate Model Comparison

Comparing multiple YOLO pose models on rondo training drill footage across all frames.

**Goal:** evaluate YOLO pose model candidates across two dimensions: player detection rate, keypoint confidence on leg joints (hips, knees, ankles). Results are combined into a composite score using subjectively weighted normalised metrics. The top two models are then visualised for qualitative assessment.

**Videos:** 6 rondo training drill clips of varying resolution and camera angles.

**Models evaluated:** 
- `yolo11n-pose` 
- `yolo11s-pose` 
- `yolo11m-pose` 
- `yolo11l-pose` 
- `yolo26m-pose`

## 0. Setup

In [ ]:
import os

%pip install ultralytics==8.4.26 -q

if not os.path.exists("football-kick-tracker"):
    !git clone https://github.com/WillEdgington/football-kick-tracker.git
else:
    !git -C football-kick-tracker pull

%pip install -e football-kick-tracker --force-reinstall -q

import sys
sys.path.insert(0, '/content/football-kick-tracker')

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO
from google.colab import drive
from pathlib import Path
from typing import Dict, List, Tuple, Literal

from pose.visualise import drawKeypoints
from pose.constants import LOWERKEYPOINTS, LOWERSKELETONPAIRS
from pose.config import CONFTHRESHOLD, LOWERCOLOURS

drive.mount('/content/drive')

## 1. Define video paths

Get paths for rondo drills from an established directory directory

In [ ]:
videoDir = Path("/content/drive/MyDrive/football-kick-tracker/data/raw/training_drills_videos")
videoPaths = sorted(videoDir.glob("*.mp4"))

assert len(videoPaths) > 0, f"No videos found in {videoDir}"
print(f"Found {len(videoPaths)} videos:")
for p in videoPaths:
    print(f"  {p.name}")

## 2. Load YOLO models

Collect models that are loadable. Store the name and the loaded model

In [ ]:
modelNames = [
    "yolo11m-pose",
    "yolo26m-pose",
    "yolo11n-pose",
    "yolo11s-pose",
    "yolo11l-pose",
]

models = []

for name in modelNames:
    try:
        model = YOLO(f"{name}.pt")
        print(f"{name} loaded")
        models.append((name, model))
    except Exception as e:
        print(f"{name} not available: {e}")

Check that models are on GPU

In [ ]:
!nvidia-smi

print(models[0][1].device)

## 3. Create save and load methods for results

In [ ]:
import json

RESULTSPATH = Path("/content/drive/MyDrive/football-kick-tracker/data/sessions/yolo_candidate_results.json")
VIDEOCACHEPATH = Path("/content/drive/MyDrive/football-kick-tracker/data/sessions/yolo_candidate_video_cache.json")

def saveJSON(data: List[Dict]|Dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmpPath = path.with_suffix(".tmp")
    try:
        with open(tmpPath, "w") as f:
            json.dump(data, f, indent=2)
        os.replace(tmpPath, path)
    except Exception as e:
        if tmpPath.exists():
            tmpPath.unlink()
        raise e

def loadJSON(path: Path) -> List[Dict]|Dict|None:
    if not path.exists():
        return None
    with open(path, "r") as f:
        return json.load(f)

## 4. Keypoint confidence from video

Collect detection and confidence metrics across all videos for a model

In [ ]:
def _keypointConfidenceOneVideo(
    model: YOLO,
    path: Path,
    name: str,
    keypointIndexes: Dict[str, int]=LOWERKEYPOINTS,
    logging: bool=True,
    cache: Dict|None=None,
    cachePath: Path|None=None,
) -> Dict[str, str|float] | None:
    cacheKey = f"{name}_{path.name}"
    if cache is not None and cacheKey in cache:
        if logging:
            print(f"[{name}] Skipping {path.name} (cached)")
        return cache[cacheKey]

    if logging:
        print(f"[{name}] Processing {path.name}...")
    results = {f"{key}_conf": 0 for key in keypointIndexes.keys()}
    
    cap = cv2.VideoCapture(str(path))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    detections = 0

    for result in model(str(path), stream=True, verbose=False):
        if result.keypoints is None:
            continue
        kps = result.keypoints.data
        if len(kps) == 0:
            continue

        for person in kps:
            detections += 1
            for key, idx in keypointIndexes.items():
                conf = float(person[idx][2])
                results[f"{key}_conf"] += conf
    
    if detections == 0:
        return None
    
    for key in results.keys():
        results[key] /= detections

    results["model"] = name
    results["detection_rate"] = detections / frames

    if cache is not None:
        cache[cacheKey] = results
        if cachePath is not None:
            saveJSON(cache, cachePath)
    
    return results

def keypointConfidenceFromVideos(
    model: YOLO,
    paths: List[Path],
    name: str,
    keypointIndexes: Dict[str, int]=LOWERKEYPOINTS,
    logging: bool=True,
    cache: Dict|None=None,
    cachePath: Path|None=None,
) -> Dict[str, str|float]|None:
    nsamples = 0
    results = {f"{key}_conf": 0 for key in keypointIndexes.keys()}
    results["detection_rate"] = 0

    for path in paths:
        videoRes = _keypointConfidenceOneVideo(
            model=model,
            path=path,
            name=name,
            keypointIndexes=keypointIndexes,
            logging=logging,
            cache=cache,
            cachePath=cachePath,
        )
        if videoRes is None:
            continue
        nsamples += 1

        for key in results.keys():
            if key not in videoRes:
                continue
            results[key] += videoRes[key]

    if nsamples == 0:
        return None
    
    for key in results.keys():
        results[key] /= nsamples

    results["model"] = name
    return results

## 5. Run all candidates

Run all YOLO candidates on keypoint detection and confidence method

In [ ]:
def runYOLOCandidates(
    models: List[Tuple[str, YOLO]],
    paths: List[Path],
    keypointIndexes: Dict[str, int]=LOWERKEYPOINTS,
    resultsPath: Path|None=None,
    logging: bool=True,
    cachePath: Path|None=None,
) -> List[Dict[str, str|float]]:
    if resultsPath is None:
        results = []
        seen = set()
    else:
        results = loadJSON(path=resultsPath) or []
        seen = {r["model"] for r in results}

    cache = loadJSON(cachePath) if cachePath is not None else None
    if cache is None and cachePath is not None:
        cache = {}

    for name, model in models:
        if name in seen:
            if logging:
                print(f"Skipping {name} (already complete)")
            continue
        print(f"Running model: {name}")
        modelRes = keypointConfidenceFromVideos(
            model=model,
            paths=paths,
            name=name,
            keypointIndexes=keypointIndexes,
            logging=logging,
            cache=cache,
            cachePath=cachePath,
        )
        if modelRes is None:
            continue
        results.append(modelRes)
        seen.add(name)
        if resultsPath:
            print(f"saving results...")
            saveJSON(
                data=results,
                path=resultsPath,
            )
            print(f"saved.")
    return results

results = runYOLOCandidates(
    models=models,
    paths=videoPaths,
    keypointIndexes=LOWERKEYPOINTS,
    resultsPath=RESULTSPATH,
    logging=True,
    cachePath=VIDEOCACHEPATH,
)

## 6. Composite Score

Create a composite score of the normalised metrics (z-score), weighted subjectively and then summed. I chose that detection rate is the most important followed by ankle confidence (the usual "kick zone") and then knee confidence (cutoff for the "kick zone"). Hip confidence is excluded as it is least relevant to kick detection.

In [ ]:
# Adapted from github.com/WillEdgington/football-torch-project (experiments/experiment_results.py as of 24/03/26)
def normaliseMetric(
    df: pd.DataFrame,
    col: str,
    method: Literal["standard", "minmax"] = "standard",
    eps: float = 1e-8,
) -> pd.Series:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found in DataFrame.")
    series = df[col].astype(float)
    if method == "standard":
        std = series.std()
        return (series - series.mean()) / max(eps, std)
    elif method == "minmax":
        low, high = series.min(), series.max()
        return (series - low) / (high - low)

def addCompositeScore(
    df: pd.DataFrame,
    weights: Dict[str, float],
    ascending: Dict[str, bool] | bool,
    colName: str = "composite_score",
    normMethod: (
        Literal["standard", "minmax"] | Dict[str, Literal["standard", "minmax"]]
    ) = "standard",
) -> pd.DataFrame:
    if isinstance(ascending, bool):
        ascending = {col: ascending for col in weights.keys()}
    if set(weights.keys()) != set(ascending.keys()):
        raise ValueError("weights and ascending must have the same keys")
    if isinstance(normMethod, dict):
        if set(weights.keys()) != set(normMethod.keys()):
            raise ValueError(
                "if normMethod is dict then it must have "
                "the same keys as weights and ascending"
            )

    df = df.copy()
    totalWeight = sum(weights.values())
    score = pd.Series(0.0, index=df.index)

    for col, weight in weights.items():
        score += (
            normaliseMetric(
                df,
                col,
                method=normMethod if not isinstance(normMethod, dict) else normMethod,
            )
            * (weight / totalWeight)
            * (-1 if ascending[col] else 1)
        )

    df[colName] = score
    return df

resultsDf = pd.DataFrame(results)

# weights are subjective
weights = {
    "detection_rate": 0.5,
    "left_ankle_conf": 0.15,
    "right_ankle_conf": 0.15,
    "left_knee_conf": 0.1,
    "right_knee_conf": 0.1,
}

resultsDf = addCompositeScore(
    df=resultsDf,
    weights=weights,
    ascending=False, # Higher is better for all metrics
)

## 7. Results table

In [ ]:
resultsDf.sort_values(
    by="composite_score",
    ascending=False,
    inplace=True,
)
print(resultsDf)

## 8. Visual comparison of top two

annotate footage using the top two models (based on composite score)

In [ ]:
annotatedVideosDir = Path("/content/drive/MyDrive/football-kick-tracker/data/sessions/annotated_videos")

def annotateVideo(
    model: YOLO,
    sourcePath: Path,
    outputPath: Path,
    keypointIndexes: Dict[str, int]=LOWERKEYPOINTS,
    skeletonPairs: List[Tuple[str, str]] = LOWERSKELETONPAIRS,
    colours: Dict[str, Tuple[int, int, int]] = LOWERCOLOURS,
    confThreshold: float = CONFTHRESHOLD,
    logging: bool=True,
) -> None:
    cap = cv2.VideoCapture(str(sourcePath))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    totalFrames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    outputPath.parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(outputPath), fourcc, fps, (width, height))
    
    for i, result in enumerate(model(str(sourcePath), stream=True, verbose=False)):
        if logging:
            print(f"Frame {i + 1}/{totalFrames}", end="\r")
        bgr = result.orig_img
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        annotated = drawKeypoints(
            frame=rgb,
            results=[result],
            keypointIndexes=keypointIndexes,
            skeletonPairs=skeletonPairs,
            colours=colours,
            confThreshold=confThreshold
        )
        bgrAnnotated = cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)
        writer.write(bgrAnnotated)

    writer.release()
    if logging:
        print(f"\nSaved to {outputPath}")

def _getOutputPaths(
    name: str,
    sourcePaths: List[Path],
    outputDir: Path,
) -> List[Path]:
    return [outputDir / name / path.name for path in sourcePaths]

def annotateVideos(
    model: YOLO,
    name: str,
    sourcePaths: List[Path],
    outputDir: Path,
    keypointIndexes: Dict[str, int]=LOWERKEYPOINTS,
    skeletonPairs: List[Tuple[str, str]] = LOWERSKELETONPAIRS,
    colours: Dict[str, Tuple[int, int, int]] = LOWERCOLOURS,
    confThreshold: float = CONFTHRESHOLD,
    logging: bool=True,
    overwrite: bool=False,
) -> List[Path]:
    outputPaths = _getOutputPaths(
        name=name,
        sourcePaths=sourcePaths,
        outputDir=outputDir,
    )

    total = len(sourcePaths)
    skipped = 0

    for i, (outputPath, sourcePath) in enumerate(zip(outputPaths, sourcePaths)):
        if outputPath.exists() and not overwrite:
            if logging:
                print(f"[{name}] Skipping {sourcePath.name} ({i + 1}/{total}) - already exists")
            skipped += 1
            continue
        if logging:
            print(f"[{name}] Annotating {sourcePath.name} ({i+1}/{total})...")
        annotateVideo(
            model=model,
            sourcePath=sourcePath,
            outputPath=outputPath,
            keypointIndexes=keypointIndexes,
            skeletonPairs=skeletonPairs,
            colours=colours,
            confThreshold=confThreshold,
            logging=logging,
        )
    
    if logging:
        completed = total - skipped
        print(f"\n[{name}] Done - {completed} annotated, {skipped} skipped.")
    return outputPaths

def _getModelByName(
    name: str,
    models: List[Tuple[str, YOLO]],
    logging: bool=True,
) -> YOLO | None:
    for n, model in models:
        if name == n:
            if logging:
                print(f"[{name}] Found model.")
            return model
    if logging:
        print(f"[{name}] Could not find model in models")
    return None

def annotateVideosForTopNModels(
    df: pd.DataFrame,
    models: List[Tuple[str, YOLO]],
    sourcePaths: List[Path],
    outputDir: Path,
    keypointIndexes: Dict[str, int]=LOWERKEYPOINTS,
    skeletonPairs: List[Tuple[str, str]] = LOWERSKELETONPAIRS,
    colours: Dict[str, Tuple[int, int, int]] = LOWERCOLOURS,
    confThreshold: float = CONFTHRESHOLD,
    logging: bool=True,
    overwrite: bool=False,
    n: int=2,
) -> Dict[str, List[Path]]:
    modelNames = df["model"]
    assert n <= len(modelNames), f"n must be less than length of df: {len(modelNames)}"
    outputPaths = {}
    
    for i in range(n):
        name = modelNames.iloc[i]
        model = _getModelByName(name=name, models=models, logging=logging)
        if model is None:
            continue
        outputPaths[name] = annotateVideos(
            model=model,
            name=name,
            sourcePaths=sourcePaths,
            outputDir=outputDir,
            keypointIndexes=keypointIndexes,
            skeletonPairs=skeletonPairs,
            colours=colours,
            confThreshold=confThreshold,
            logging=logging,
            overwrite=overwrite
        )
    return outputPaths

annotatedVideoPaths = annotateVideosForTopNModels(
    df=resultsDf,
    models=models,
    sourcePaths=videoPaths,
    outputDir=annotatedVideosDir,
    keypointIndexes=LOWERKEYPOINTS,
    skeletonPairs=LOWERSKELETONPAIRS,
    colours=LOWERCOLOURS,
    confThreshold=CONFTHRESHOLD,
    logging=True,
    overwrite=False,
    n=2, # only the two best models
)

Plot frames from each video

In [ ]:
def _getMiddleFrameFromPath(
    path: Path
) -> np.ndarray | None:
    if not path.exists():
        return None
    cap = cv2.VideoCapture(path)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    mid = frames >> 1
    cap.set(cv2.CAP_PROP_POS_FRAMES, mid)
    ret, frame = cap.read()
    cap.release()
    
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None

def plotAnnotatedFramesFromPaths(
    pathsDict: Dict[str, List[Path]],
) -> plt.Figure:
    cols = len(pathsDict.keys()) 
    rows = max(len(paths) for paths in pathsDict.values())
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 4 * rows))

    for j, (name, paths) in enumerate(pathsDict.items()):
        for i, path in enumerate(paths):
            frame = _getMiddleFrameFromPath(
                path=path
            )
            axes[i, j].imshow(frame)
            axes[i, j].set_title(f"{name} (video: {path.name})")
            axes[i, j].axis("off")
    
    plt.suptitle("Annotated Frame Comparison (multiple videos) (orange=hip, blue=knee, green=ankle)", y=1.01)
    plt.tight_layout()
    plt.show()
    return fig

fig = plotAnnotatedFramesFromPaths(pathsDict=annotatedVideoPaths)

## 9. Observations

**Experiment Structure:**
- Detection rate per frame per video
- Confidence of leg keypoints per detection per video
- Both metrics summed into a subjectively weighted composite score for easy ranking
- Check annotated videos for any systematic failures

**Notes:**
- YOLO11(n)(s)(l)(m) all performed much better than YOLO26m (composite score)
- The two best models over the 6 clips (composite score) were:
  - YOLO11l: detection rate (~5.5), ankle confidence (~0.93), knee confidence (~0.97)
  - YOLO11m: detection rate (~4.9), ankle confidence (~0.94), knee confidence (~0.97)
- From analysing the top two models visually over the 6 clips:
  - I was more impressed with YOLO11l's detection ability (it particularly performed better on subjects that were "mid-action")
  - Both failed with subjects that were far away (this could be further emphasised due to the low quality of footage I have used)
- Although not formally tested, YOLO11l seemed to have the slowest inference speed (This could become a factor when Real-time inference or greater accessibility become a priority)

**Decision:**
- YOLO11l is the model that will be used going forward (as opposed to YOLO11m)
- Future experimentation is needed to test non-YOLO models (ViTPose, RTMPose), inference speed, and better test samples